In [1]:
import pandas as pd

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [2]:
df=pd.read_csv('11march.csv')
df["Username"]

0         alluredbynature
1           frolicking.by
2        meenakshi.saini_
3              indie.skin
4         phalgunisarkar_
              ...        
5984      archiherbals.in
5985      shreem.organics
5986             ura.care
5987      ttrunk_organics
5988    himalaya_facecare
Name: Username, Length: 5989, dtype: str

In [3]:
df["Username"].to_csv("usernames.txt", index=False, header=False)

In [1]:
!pip install selenium webdriver-manager pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached webdriver_manager-4.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached webdriver_manager-4.1.2-py3-none-any.whl (32 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [webdriver-manager]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# instagram_selenium_scraper.py
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import json
import logging
import os
import re
from typing import Dict, List, Optional
from datetime import datetime

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class InstagramScraper:
    """Instagram profile scraper using Selenium"""
    
    def __init__(self, headless: bool = False):
        """
        Initialize the Chrome driver
        
        Args:
            headless: Run in headless mode (no browser window)
        """
        self.options = Options()
        
        if headless:
            self.options.add_argument('--headless')
        
        # Additional options to avoid detection
        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument('--disable-gpu')
        self.options.add_argument('--window-size=1920,1080')
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        
        # User agent to appear like a real browser
        self.options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        
        # Initialize driver
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        self.wait = WebDriverWait(self.driver, 10)
        
    def get_user_data(self, username: str) -> Dict:
        """
        Scrape Instagram profile data for a given username
        
        Args:
            username: Instagram username (without @)
            
        Returns:
            Dictionary with user data
        """
        url = f"https://www.instagram.com/{username}/"
        logger.info(f"🌐 Scraping: {url}")
        
        try:
            # Navigate to profile
            self.driver.get(url)
            time.sleep(5)  # Wait for page load
            
            # Wait for page to load
            try:
                self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            except TimeoutException:
                logger.warning("Page load timeout, continuing anyway...")
            
            # Try to detect if profile exists
            page_text = self.driver.page_source
            if "Sorry, this page isn't available" in page_text:
                logger.error(f"❌ Profile not found: {username}")
                return {"username": username, "error": "Profile not found", "timestamp": datetime.now().isoformat()}
            
            # Extract data using robust methods
            data = {
                "username": username,
                "full_name": self._get_username(),
                "followers": self._get_followers(),
                "following": self._get_following(),
                "bio": self._get_bio(),
                "portfolio_link": self._get_portfolio_link(),
                "scraped_at": datetime.now().isoformat()
            }
            
            logger.info(f"✅ Successfully scraped: {username}")
            return data
            
        except Exception as e:
            logger.error(f"❌ Error scraping {username}: {e}")
            return {
                "username": username, 
                "error": str(e),
                "scraped_at": datetime.now().isoformat()
            }
    
    def _get_username(self) -> Optional[str]:
        """Get username from profile"""
        selectors = [
            '//header//h2//span',
            '//header//section[2]//div//div//div//div//div//h2//span',
            '//*[@id="mount_0_0_ND"]//header//h2//span',
            'header h2 span',
            '//span[@class="x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp x1s688f x5n08af"]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    text = element.text.strip()
                    if text and len(text) > 1 and not text.isdigit():
                        # Make sure it's not follower/following count
                        if 'follower' not in text.lower() and 'following' not in text.lower():
                            return text
            except:
                continue
        return None
    
    def _get_followers(self) -> Optional[str]:
        """Get followers count from profile"""
        selectors = [
            '//header//ul//li[1]//span',
            '//header//section[3]//ul//li[1]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[1]//span',
            'header ul li:first-child span',
            '//li[1]//span[contains(@class, "_ac2a")]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    # Clean up - remove commas and convert to int
                    return text.replace(',', '')
            except:
                continue
        return None
    
    def _get_following(self) -> Optional[str]:
        """Get following count from profile"""
        selectors = [
            '//header//ul//li[2]//span',
            '//header//section[3]//ul//li[2]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[2]//span',
            'header ul li:nth-child(2) span',
            '//li[2]//span[contains(@class, "_ac2a")]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text.replace(',', '')
            except:
                continue
        return None
    
    def _get_bio(self) -> Optional[str]:
        """
        Get bio from profile - USING YOUR EXACT SELECTORS
        """
        # Wait a moment for bio to load
        time.sleep(2)
        
        # Method 1: Using the exact class you provided for bio header
        bio_selectors = [
            # Your exact class from the bio header
            'span[class*="x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp x1s688f x5n08af"]',
            
            # The bio class you provided
            'span._ap3a._aaco._aacu._aacx._aad7._aade',
            'span._ap3a',
            'span._aade',
            
            # Using partial class match
            'span[class*="_ap3a"]',
            'span[class*="_aaco"]',
            'span[class*="_aad7"]',
            
            # Your XPath based on the classes
            '//span[@class="_ap3a _aaco _aacu _aacx _aad7 _aade"]',
            '//span[contains(@class, "_ap3a") and contains(@class, "_aaco")]',
            '//span[contains(@class, "_aacu")]',
            
            # Generic selectors
            'header section:last-child span._ap3a',
            'header div[dir="auto"] span._ap3a',
            'header ._ap3a._aaco',
            
            # Original selectors that might work
            '//header//div[contains(@class, "_aa_c")]//span//div//span',
            '//header//div[contains(@class, "_ab_9")]//span',
            '//header//section[4]//div//span//div//span',
            
            # CSS Selectors
            'header div[class*="_aa_c"] span div span',
            'header div[class*="_ab_9"] span',
            'header section:last-child div span div span',
            'header ._aa_c span div span',
        ]
        
        for selector in bio_selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    text = element.text.strip()
                    if text and len(text) > 1:
                        # Make sure it's bio (not follower/following count)
                        if 'follower' not in text.lower() and 'following' not in text.lower():
                            if not text.replace(',', '').isdigit():
                                logger.info(f"✅ Found bio: {text[:100]}...")
                                return text
            except Exception as e:
                continue
        
        # Method 2: Search for bio using div with dir="auto"
        try:
            bio_elements = self.driver.find_elements(By.CSS_SELECTOR, 'div[dir="auto"] span[class*="_ap3a"]')
            for element in bio_elements:
                text = element.text.strip()
                if text and len(text) > 5:
                    if 'follower' not in text.lower() and 'following' not in text.lower():
                        if not text.replace(',', '').isdigit():
                            return text
        except:
            pass
        
        # Method 3: Get all text from header and find bio
        try:
            header = self.driver.find_element(By.TAG_NAME, 'header')
            all_text = header.text.split('\n')
            
            # Find lines that look like bio (not numbers, not just usernames)
            for line in all_text:
                line = line.strip()
                if line and len(line) > 5:
                    # Skip if it's just numbers (followers/following)
                    if not line.replace(',', '').isdigit():
                        # Skip if it's a username pattern
                        if not line.startswith('@'):
                            # Skip if it's just a name
                            if ' ' in line and len(line.split()) > 1:
                                return line
        except:
            pass
        
        return None
    
    def _get_portfolio_link(self) -> Optional[str]:
        """
        Get portfolio link from profile - USING YOUR EXACT SELECTORS
        """
        # Wait a moment for link to load
        time.sleep(2)
        
        # Method 1: Using your exact div class
        link_selectors = [
            # Your exact class from the portfolio link
            'div._ap3a._aaco._aacw._aacz._aada._aade',
            'div[class*="_ap3a _aaco _aacw _aacz _aada _aade"]',
            'div._ap3a._aaco',
            'div._aacw',
            
            # Using partial class match
            'div[class*="_ap3a"][class*="_aaco"]',
            'div[class*="_aacw"]',
            'div[class*="_aacz"]',
            
            # Your XPath based on the classes
            '//div[@class="_ap3a _aaco _aacw _aacz _aada _aade"]',
            '//div[contains(@class, "_ap3a") and contains(@class, "_aaco")]',
            
            # Find by dir="auto" attribute
            'div[dir="auto"]._ap3a._aaco',
            'div[dir="auto"][class*="_ap3a"]',
            
            # Existing selectors
            '//header//a[contains(@href, "http")]',
            '//header//div[contains(@class, "_ac2b")]//a',
            '//header//section[4]//div//div[3]//div//a',
            
            # CSS Selectors
            'header a[href*="http"]',
            'header div[class*="_ac2b"] a',
            'header section:last-child a',
            'header ._aa_c a',
        ]
        
        for selector in link_selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    # Try to get href first
                    href = element.get_attribute('href')
                    if href and ('http' in href or 'https' in href):
                        if 'instagram.com' not in href:
                            logger.info(f"✅ Found portfolio link: {href}")
                            return href
                    
                    # If no href, get text (like your example: nyk0.page.link/m5n923WDNi69Zx1V8)
                    text = element.text.strip()
                    if text and '.' in text and ' ' not in text:
                        # Check if it looks like a URL
                        if 'http' not in text and len(text) > 5:
                            # Add http if missing
                            if not text.startswith('http'):
                                full_url = f"https://{text}"
                                logger.info(f"✅ Found portfolio link: {full_url}")
                                return full_url
                        elif text.startswith('http'):
                            logger.info(f"✅ Found portfolio link: {text}")
                            return text
            except Exception as e:
                continue
        
        # Method 2: Search for text that looks like a URL
        try:
            all_elements = self.driver.find_elements(By.XPATH, '//*')
            for element in all_elements:
                text = element.text.strip()
                if text and '.' in text and ' ' not in text and len(text) > 5:
                    # Check if it looks like a URL or shortened link
                    if 'http' in text or 'bit.ly' in text or 'link' in text or '.page.link' in text:
                        if 'instagram.com' not in text:
                            if not text.startswith('http'):
                                return f"https://{text}"
                            return text
        except:
            pass
        
        return None
    
    def bulk_scrape_with_append(self, usernames: List[str], output_csv: str, delay: float = 3.0) -> List[Dict]:
        """
        Bulk scrape and append results instantly to CSV
        
        Args:
            usernames: List of Instagram usernames
            output_csv: Output CSV filename
            delay: Delay between requests in seconds
            
        Returns:
            List of user data dictionaries
        """
        results = []
        success_count = 0
        fail_count = 0
        
        logger.info(f"🚀 Starting bulk scrape for {len(usernames)} users...")
        logger.info(f"📁 Saving to: {output_csv}")
        
        # Initialize CSV file with headers if it doesn't exist
        if not os.path.exists(output_csv):
            empty_df = pd.DataFrame(columns=['username', 'full_name', 'followers', 'following', 'bio', 'portfolio_link', 'scraped_at', 'error'])
            empty_df.to_csv(output_csv, index=False, encoding='utf-8')
            logger.info(f"📄 Created new CSV file: {output_csv}")
        
        for i, username in enumerate(usernames, 1):
            logger.info(f"📊 {i}/{len(usernames)}: Scraping {username}")
            
            # Scrape user
            data = self.get_user_data(username)
            results.append(data)
            
            # Immediately append to CSV
            self._append_to_csv(data, output_csv)
            
            # Track success/failure
            if 'error' in data and data['error']:
                fail_count += 1
                logger.info(f"⚠️  {username}: FAILED")
            else:
                success_count += 1
                logger.info(f"✅ {username}: SUCCESS")
                logger.info(f"   Bio: {data.get('bio', 'N/A')[:50]}...")
                logger.info(f"   Portfolio: {data.get('portfolio_link', 'N/A')}")
            
            # Rate limiting
            if i < len(usernames):
                time.sleep(delay)
            
            # Save progress backup every 5 users
            if i % 5 == 0:
                self._save_progress_backup(results, f"backup_progress_{i}.json")
                logger.info(f"💾 Progress backup saved at {i} users")
        
        logger.info(f"✅ Completed scraping {len(results)} users")
        logger.info(f"📊 Success: {success_count}, Failed: {fail_count}")
        
        return results
    
    def _append_to_csv(self, data: Dict, filename: str):
        """Append a single record to CSV instantly"""
        try:
            row = {
                'username': data.get('username', ''),
                'full_name': data.get('full_name', ''),
                'followers': data.get('followers', ''),
                'following': data.get('following', ''),
                'bio': data.get('bio', ''),
                'portfolio_link': data.get('portfolio_link', ''),
                'scraped_at': data.get('scraped_at', datetime.now().isoformat()),
                'error': data.get('error', '')
            }
            
            df_row = pd.DataFrame([row])
            df_row.to_csv(filename, mode='a', header=False, index=False, encoding='utf-8')
            
        except Exception as e:
            logger.error(f"❌ Error appending to CSV: {e}")
    
    def _save_progress_backup(self, data: List[Dict], filename: str):
        """Save progress backup"""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            logger.error(f"❌ Error saving backup: {e}")
    
    def close(self):
        """Close the Chrome driver"""
        self.driver.quit()


# ===================== MAIN EXECUTION =====================

def main():
    """Main execution function"""
    input_csv = "11march.csv"
    username_column = "Username"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv = f"instagram_scraped_{timestamp}.csv"
    
    print("\n" + "="*60)
    print("📱 INSTAGRAM BULK SCRAPER (FIXED WITH YOUR SELECTORS)")
    print("="*60)
    
    # Load usernames from CSV
    try:
        df = pd.read_csv(input_csv)
        usernames = df[username_column].dropna().astype(str).tolist()
        usernames = [u.strip() for u in usernames if u.strip()]
        print(f"📄 Loaded {len(usernames)} usernames from {input_csv}")
        print(f"📋 First 5: {', '.join(usernames[:5])}")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    if not usernames:
        print("❌ No usernames found!")
        return
    
    response = input("\nContinue? (y/n): ").strip().lower()
    if response != 'y':
        print("❌ Cancelled")
        return
    
    scraper = InstagramScraper(headless=False)
    
    try:
        data = scraper.bulk_scrape_with_append(usernames, output_csv, delay=3.0)
        
        print("\n" + "="*60)
        print("📊 SUMMARY")
        print("="*60)
        
        success = sum(1 for d in data if 'error' not in d or not d.get('error'))
        failed = len(data) - success
        
        print(f"✅ Success: {success}")
        print(f"❌ Failed: {failed}")
        print(f"📁 Output: {output_csv}")
        
        # Show sample of scraped data
        print("\n📋 Sample Output:")
        for user in data[:3]:
            if 'error' not in user or not user.get('error'):
                print(f"\n✅ {user.get('username')}")
                print(f"   Bio: {user.get('bio', 'N/A')[:100]}...")
                print(f"   Portfolio: {user.get('portfolio_link', 'N/A')}")
        
    except KeyboardInterrupt:
        print("\n⏹️ Interrupted - Data saved so far!")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        scraper.close()


if __name__ == "__main__":
    main()


📱 INSTAGRAM BULK SCRAPER (FIXED WITH YOUR SELECTORS)
📄 Loaded 5989 usernames from 11march.csv
📋 First 5: alluredbynature, frolicking.by, meenakshi.saini_, indie.skin, phalgunisarkar_


2026-09-06 11:08:38,409 - INFO - ====== WebDriver manager ======
2026-09-06 11:08:38,458 - INFO - Driver [/home/itm/.wdm/drivers/chromedriver/linux64/148.0.7778.178/chromedriver-linux64/chromedriver] found in cache by browser version
2026-09-06 11:08:39,461 - INFO - 🚀 Starting bulk scrape for 5989 users...
2026-09-06 11:08:39,462 - INFO - 📁 Saving to: instagram_scraped_20260906_110836.csv
2026-09-06 11:08:39,464 - INFO - 📄 Created new CSV file: instagram_scraped_20260906_110836.csv
2026-09-06 11:08:39,465 - INFO - 📊 1/5989: Scraping alluredbynature
2026-09-06 11:08:39,465 - INFO - 🌐 Scraping: https://www.instagram.com/alluredbynature/
2026-09-06 11:08:55,547 - INFO - ✅ Found bio: Allured by Nature®...
2026-09-06 11:08:57,589 - INFO - ✅ Found portfolio link: https://wa.me/message/4S6JAFR6WQXJO1
2026-09-06 11:08:57,590 - INFO - ✅ Successfully scraped: alluredbynature
2026-09-06 11:08:57,594 - INFO - ✅ alluredbynature: SUCCESS
2026-09-06 11:08:57,595 - INFO -    Bio: Allured by Nature®...

❌ Error: 'NoneType' object is not subscriptable
